In [3]:
#######################################################
##### Cumulative-Microscopy-Python
#######################################################
### Custom Python script designed for quantitative analysis of multiplex imaging data acquired through cumulative microscopy.
### The pipeline processes multi‑cycle image datasets, performs image segmentation and quantification, calculates background correction
### using control samples, and outputs cleaned, annotated results.

### Please check README file before running the script 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import filters
from skimage import segmentation
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from skimage import morphology
from skimage import measure
from skimage import restoration
from skimage.measure import blur_effect
import tifffile
import cv2

import shutil
import os
from os import walk
from pathlib import Path
import time
import sys
import winsound

In [4]:
########## PART 1 ###########
#### Image quantification ######
###############################

startTime = time.time()

dir_path='Your_Input_directory' # Specify input directory
dir_save ='Your_Output_directory' #  Specify output directory

####### Settings for Test mode #######
##### Use test mode (testMode = 1) to adjust segmentation parameters and visually inspect results.
##### Once optimal settings are defined, switch back to running mode (testMode = 0) to process the full dataset.

testMode=0   #### 1 = test mode, 0 = full analysis
Cfolder = 'c1'  # Cycle for test mode
Sfolder = 'C04' # SampleID for test mode
slForTest = 3   # Site for test mode

Dapi_w = 'w1'
W2 = 'w2'
W3 = 'w3'
W4 = 'w4'
file_ext_dapi = Dapi_w+'.TIF'

hole_size = 50
min_size = 250  
max_size = 900 
min_dist = 20 #20
auto_min_dist = 0

focusK = 0.8 ##### skip the object if not in focus (0 - perfect focus, 1 - not in focus)
crowdK = 400 #### skip the image if too crowded (more than 400 objects)


######### background #####
def subtract_background(nuclei_image, radius=50):
    structuring_element = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (radius, radius))
    n_background = cv2.morphologyEx(nuclei_image, cv2.MORPH_OPEN, structuring_element)
    nuclei_ab = cv2.subtract(nuclei_image, n_background)

    ######### threshold #####
    denoised = nuclei_ab
    
    ### Option 1 - Auto
    lii_thresholded = denoised > filters.threshold_otsu(denoised) ## Auto thresholding Outsu

    ### Option 2 - Manual thresholding
    #ManualThreshold = 550 
    #lii_thresholded = denoised >  ManualThreshold
    
    ######### END threshold #####
    
    return lii_thresholded


######### END background #####

####### END Settings for Test mode #######

res = []
tit_dapi = []


if testMode ==1:
    dir_test = dir_path+'/'+Sfolder+'/'+Cfolder
    for (dir_test, dir_names, file_names) in walk(dir_test):
        res.extend(file_names)
    jj=0
    for j in range(len(res)):
        file=res[j]
        if file_ext_dapi in file:
            #tit.extend(file)
            tit_dapi.insert(jj, file)
            jj+=1
    test_file_name=dir_test+'/'+tit_dapi[slForTest]
    
    nuclei = tifffile.imread(test_file_name)
    
    print("shape: {}".format(nuclei.shape))
    print("dtype: {}".format(nuclei.dtype))
    print("range: ({}, {})".format(np.min(nuclei), np.max(nuclei)))
    
    
    
    li_thresholded = subtract_background(nuclei)
    
    remove_holes = morphology.remove_small_holes(
    li_thresholded, hole_size
    )
    
    remove_objects_1 = morphology.remove_small_objects(
    remove_holes, min_size
    )
    
    remove_objects_2 = morphology.remove_small_objects(
    remove_objects_1, max_size
    )
    
    remove_objects = remove_objects_1.astype(np.uint8) - remove_objects_2.astype(np.uint8)
    distance = ndi.distance_transform_edt(remove_objects)
    coords = peak_local_max(distance, footprint=np.ones((3, 3)), min_distance=min_dist, labels=remove_objects)
    mask = np.zeros(distance.shape, dtype=bool)
    mask[tuple(coords.T)] = True
    markers, _ = ndi.label(mask)
    labels =segmentation.watershed(-distance, markers, mask=remove_objects)
    
    segmented_padded = np.pad(
        labels,
        ((0, 0), (0, 0)),
        mode='constant',
        constant_values=0,
    )
    interior_labels = segmentation.clear_border(segmented_padded)

    regionprops = measure.regionprops(interior_labels, intensity_image=nuclei)

    
    Min_l = []
    
    jjj=0
    for i in range(0, len(regionprops)):
        if regionprops[i].area > min_size:
            Min_l.insert(jjj, regionprops[i].minor_axis_length)
            jjj+=1
            
    
    
    if auto_min_dist ==1:
        min_dist_a=min_dist
                
        if len(Min_l)>0:
            min_dist_a=round(np.percentile(Min_l,2))
            
        coords = peak_local_max(distance, footprint=np.ones((3, 3)), min_distance=min_dist_a, labels=remove_objects)
        mask = np.zeros(distance.shape, dtype=bool)
        mask[tuple(coords.T)] = True
        markers, _ = ndi.label(mask)
        labels =segmentation.watershed(-distance, markers, mask=remove_objects)
    
        segmented_padded = np.pad(
            labels,
            ((0, 0), (0, 0)),
            mode='constant',
            constant_values=0,
        )
        interior_labels = segmentation.clear_border(segmented_padded)

        
    mark_boundaries = segmentation.mark_boundaries(nuclei, interior_labels, color=(1, 1, 0), outline_color=None, mode='outer', background_label=0)
    
    fig, axes = plt.subplots(ncols=2, nrows=2, figsize=(18, 18), sharex=True, sharey=True)
    ax = axes.ravel()

    ax[0].imshow(li_thresholded, cmap=plt.cm.gray)
    ax[0].set_title('DAPI')
    ax[1].imshow(remove_objects, cmap=plt.cm.gray)
    ax[1].set_title('Mask')
    ax[2].imshow(-interior_labels, cmap=plt.cm.nipy_spectral) #Paired (labels, cmap='tab20b')
    ax[2].set_title('Objects')
    ax[3].imshow(mark_boundaries) ## 
    ax[3].set_title('clear objects from the image border')

    for a in ax:
        a.set_axis_off()

    fig.tight_layout()
    plt.show()
    
    
    print ('Test Mode')
    sys.exit('Test mode is over')


########################################################################
########################################################################
############################# MAIN PART ################################

#######################################################################


os.chdir(dir_path)
Samples = next(os.walk('.'))[1]

for numSam in range(0, len(Samples)):
    os.chdir(dir_path+'/'+Samples[numSam])
    Cyc=next(os.walk('.'))[1]
    
    for numCyc in range(0, len(Cyc)):
        
        dir=dir_path+'/'+Samples[numSam]+'/'+Cyc[numCyc]
        
        res = []
        tit_dapi = []
        
        for (dir, dir_names, file_names) in walk(dir):
            res.extend(file_names)
        jj=0
        for j in range(len(res)):
            file=res[j]
            if file_ext_dapi in file:
                tit_dapi.insert(jj, file)
                jj+=1
                
        file_dapi=dir+'/'+tit_dapi[0]
        file_w2 = dir+'/'+tit_dapi[0].replace(Dapi_w, W2)
        file_w3 = dir+'/'+tit_dapi[0].replace(Dapi_w, W3)
        file_w4 = dir+'/'+tit_dapi[0].replace(Dapi_w, W4)
                 
        ch_w2=0
        ch_w3=0
        ch_w4=0
        
        if os.path.isfile(file_w2):
            ch_w2=1
        if os.path.isfile(file_w3):
            ch_w3=1
        if os.path.isfile(file_w4):
            ch_w4=1
        
        Sample_name = []
        Cell_num =[]
        Slice = []
        centroid_X= []
        centroid_Y= []
        Area= []
        intensity_mean_dapi= []
        intensity_mean_w2 = []
        intensity_mean_w3 = []
        intensity_mean_w4 = []
            
        kkk=0        
        
        for k in range(len(tit_dapi)):  #### main for
            
            SliceN = []
            SliceN = tit_dapi[k].split('_')
            s =  int(SliceN[2].replace('s', ''))
        
            file_dapi=dir+'/'+tit_dapi[k]
            nuclei = tifffile.imread(file_dapi)
            
            if ch_w2==1:
                file_w2 = dir+'/'+tit_dapi[k].replace(Dapi_w, W2)
                img_w2 = tifffile.imread(file_w2)
            
            if ch_w3==1:
                file_w3 = dir+'/'+tit_dapi[k].replace(Dapi_w, W3)
                img_w3 = tifffile.imread(file_w3)
                
            if ch_w4==1:
                file_w4 = dir+'/'+tit_dapi[k].replace(Dapi_w, W4)
                img_w4 = tifffile.imread(file_w4)
                            
                    
                        
            li_thresholded = subtract_background(nuclei)
            
            remove_holes = morphology.remove_small_holes(li_thresholded, hole_size)
    
            remove_objects_1 = morphology.remove_small_objects(remove_holes, min_size)
    
            remove_objects_2 = morphology.remove_small_objects(remove_objects_1, max_size)
    
            remove_objects = remove_objects_1.astype(np.uint8) - remove_objects_2.astype(np.uint8)
            distance = ndi.distance_transform_edt(remove_objects)
            coords = peak_local_max(distance, footprint=np.ones((3, 3)), min_distance=min_dist, labels=remove_objects)
            mask = np.zeros(distance.shape, dtype=bool)
            mask[tuple(coords.T)] = True
            markers, _ = ndi.label(mask)
            labels =segmentation.watershed(-distance, markers, mask=remove_objects)
    
            segmented_padded = np.pad(labels, ((0, 0), (0, 0)), mode='constant', constant_values=0,)
            
            interior_labels = segmentation.clear_border(segmented_padded)

            regionprops = measure.regionprops(interior_labels, intensity_image=nuclei)

            if auto_min_dist ==1:
            
                Min_l = []
    
                jjj=0
                for i in range(0, len(regionprops)):
                    if regionprops[i].area > min_size:
                        Min_l.insert(jjj, regionprops[i].minor_axis_length)
                        jjj+=1
                        
                
                min_dist_a=min_dist
                
                if len(Min_l)>0:
                    min_dist_a=round(np.percentile(Min_l,2))
                
                coords = peak_local_max(distance, footprint=np.ones((3, 3)), min_distance=min_dist_a, labels=remove_objects)
                mask = np.zeros(distance.shape, dtype=bool)
                mask[tuple(coords.T)] = True
                markers, _ = ndi.label(mask)
                labels =segmentation.watershed(-distance, markers, mask=remove_objects)
    
                segmented_padded = np.pad(labels, ((0, 0), (0, 0)),mode='constant', constant_values=0,)
                
                interior_labels = segmentation.clear_border(segmented_padded)

 
            if len(regionprops)<crowdK: # skip the image if too crowded
                data_dapi = measure.regionprops(interior_labels, intensity_image=nuclei)
            
                if ch_w2==1:
                    data_w2 = measure.regionprops(interior_labels, intensity_image=img_w2)
                if ch_w3==1:
                    data_w3 = measure.regionprops(interior_labels, intensity_image=img_w3)
                if ch_w4==1:
                    data_w4 = measure.regionprops(interior_labels, intensity_image=img_w4)
            
                
                for kk in range(0, len(data_dapi)):
                    if data_dapi[kk].area > min_size:
                        ##### checking if object is in focus
                        c_x=round(data_dapi[kk].centroid[1] - data_dapi[kk].major_axis_length/2)
                        if c_x<0:
                            c_x=0
                        c_y=round(data_dapi[kk].centroid[0] - data_dapi[kk].major_axis_length/2)
                        if c_y<0:
                            c_y=0
                        c_h = round(data_dapi[kk].major_axis_length)
                        
                        c_w = round(data_dapi[kk].major_axis_length)
                        
                        crop_nuc = nuclei[c_y:c_y+c_h, c_x:c_x+c_w]

                        if blur_effect(crop_nuc)<focusK: ##### checking if object is in focus
                            Cell_num.insert(kkk, kkk+1)
                            Sample_name.insert(kkk, Samples[numSam])
                            Slice.insert(kkk, s)
                            centroid_X.insert(kkk, data_dapi[kk].centroid[1])
                            centroid_Y.insert(kkk, data_dapi[kk].centroid[0])
                            Area.insert(kkk, data_dapi[kk].area)
                            intensity_mean_dapi.insert(kkk, data_dapi[kk].mean_intensity)
                            if ch_w2==1:
                                intensity_mean_w2.insert(kkk, data_w2[kk].mean_intensity)
                            if ch_w3==1:
                                intensity_mean_w3.insert(kkk, data_w3[kk].mean_intensity)
                            if ch_w4==1:
                                intensity_mean_w4.insert(kkk, data_w4[kk].mean_intensity)
                            kkk+=1
            
        tb = pd.DataFrame(columns=['Sample', 'Slice', 'Num', 'X', 'Y', 'Area', 'DAPIMean'])
        tb['Sample'] = Sample_name
        tb['Slice'] = Slice
        tb['Num'] = Cell_num
        tb['X'] = centroid_X
        tb['Y'] = centroid_Y
        tb['Area'] = Area
        tb['DAPIMean'] = intensity_mean_dapi
        if ch_w2==1:
            tb.insert(loc=7, column=W2+'_'+Cyc[numCyc], value=intensity_mean_w2, allow_duplicates=True)
        if ch_w3==1:
            tb.insert(loc=8, column=W3+'_'+Cyc[numCyc], value=intensity_mean_w3, allow_duplicates=True)
        if ch_w4==1:
            tb.insert(loc=9, column=W4+'_'+Cyc[numCyc], value=intensity_mean_w4, allow_duplicates=True)
        
        
        tb_name='Initial_Table_'+Samples[numSam]+'_'+Cyc[numCyc]+'.csv'
        filepath = Path(dir_save+'/'+Samples[numSam]+'/'+tb_name)
        filepath.parent.mkdir(parents=True, exist_ok=True)
            
        tb.to_csv(filepath)    
            

        
       
    tt=(time.time() - startTime)/60
    ttr = (numSam+1)*100/(len(Samples)+1)
    
    print('Completed: ', Samples[numSam], 'Total progress: ', round(ttr, 2), '%. Time:', round(tt), 'min.')
    
    
print ('DONE! Total Time:', round(tt), 'min.' )
frequency = 800  # Set Frequency To 2500 Hertz
duration = 2000  # Set Duration To 1000 ms == 1 second
winsound.Beep(frequency, duration)
    

Completed:  B04 Total progress:  14.29 %. Time: 1 min.
Completed:  C04 Total progress:  28.57 %. Time: 1 min.
Completed:  D04 Total progress:  42.86 %. Time: 2 min.
Completed:  E04 Total progress:  57.14 %. Time: 3 min.
Completed:  F04 Total progress:  71.43 %. Time: 3 min.
Completed:  G04 Total progress:  85.71 %. Time: 3 min.
DONE! Total Time: 3 min.


In [5]:
########## PART 2 ###########
#### Matching cells from different cycles ######
###############################

from skimage.registration import phase_cross_correlation

dir_save_data = dir_save 


Cycle_I = 'c'

os.chdir(dir_save) ### os.chdir(dir_save)
Samples = next(os.walk('.'))[1]

os.chdir(dir_path+'/'+Samples[0])
Cyc=next(os.walk('.'))[1]

dir=dir_path+'/'+Samples[0]+'/'+Cyc[0]
res = []
tit_dapi = []
        
for (dir, dir_names, file_names) in walk(dir):
    res.extend(file_names)
jj=0
for j in range(len(res)):
    file=res[j]
    if file_ext_dapi in file:
        tit_dapi.insert(jj, file)
        jj+=1
                
for numSamtb in range(0, len(Samples)): 
    dir_tb = dir_save+'/'+Samples[numSamtb]
    
    initial_tb = []
    
    for (dir_tb, dir_names, file_names) in walk(dir_tb):
            initial_tb.extend(file_names)
    
    tb1_file_name = 'Initial_Table_'+Samples[numSamtb]+'_'+Cycle_I+str(1)+'.csv'
    tb1_file_p = dir_tb+'/'+tb1_file_name
    
    tb1= pd.read_csv(tb1_file_p) 
    
    if len(initial_tb)>1:
             
        shift1 = 20
######################################################
###########################################################
        
        tb3=[]   
        for ii in range(0, len(tb1)): #for ii in range(0, len(tb1)) range(0, 2)
            #Col_tb1=[]
            Col_tb1=tb1.iloc[[ii]]
            Col_tb2=[]
            cell_match=1
            for iii in range(1, len(initial_tb)):  
                tb2_file_name = 'Initial_Table_'+Samples[numSamtb]+'_'+Cycle_I+str(iii+1)+'.csv'
                tb2_file_p = dir_tb+'/'+tb2_file_name
                tb2= pd.read_csv(tb2_file_p)
            
###############################################################
##### Cell matching.
#### 1. First, check cells that are at a certain distance (to avoid an error if a cell is missing in the next cycle).
##### 2. Choose the closest cell from different cycles.            
###### #########################################################
                
                Col_tb2 = (
                    tb2[(tb2['Slice'] == tb1['Slice'][ii]) &
                        (abs(tb2['X'] - tb1['X'][ii]) < shift1) &
                        (abs(tb2['Y'] - tb1['Y'][ii]) < shift1)]
                    .assign(dist=lambda d: ((d['X'] - tb1['X'][ii])**2 + (d['Y'] - tb1['Y'][ii])**2)**0.5)
                    .nsmallest(1, 'dist')
                )

###############################################################
                ##### END Cell matching
################################################################
                
                
                if len(Col_tb2)==1: 
                    Col_tb2 = Col_tb2.reset_index(drop=True)
                    Col_tb1 = Col_tb1.reset_index(drop=True)
                    frames = [Col_tb1, Col_tb2]
                    Col_tb1 = pd.concat(frames, axis=1)#.ignore_index(drop=True)
                    Col_tb2 =[]
                else:
                    cell_match=0
                    break
            
            
            if cell_match>0:
                if len(tb3)==0:
                    tb3=Col_tb1
                else:
                    big_frames = [tb3, Col_tb1]
                    tb3 = pd.concat(big_frames) #.reset_index(drop=True)
            
        tb_tmp='tmp_'+Samples[numSamtb]+'.csv'
        filepath_tmp = Path(dir_save_data+'/tmp/'+tb_tmp)
        filepath_tmp.parent.mkdir(parents=True, exist_ok=True)
    
        tb3.drop('Unnamed: 0', inplace=True, axis=1, errors='ignore')
        tb4=tb3.reset_index()    
        tb4.drop('index', inplace=True, axis=1, errors='ignore')
        tb4.to_csv(filepath_tmp)
        tb3=[]
        tb4.drop('Slice', inplace=True, axis=1, errors='ignore')
        tb4.drop('Num', inplace=True, axis=1, errors='ignore')
        tb4.drop('X', inplace=True, axis=1, errors='ignore')
        tb4.drop('Y', inplace=True, axis=1, errors='ignore')
        
        cols = []
        count = 1
        for column in tb4.columns:
            if column == 'Sample':
                if  count > 1:
                    cols.append(f'Sample_R')
                else:
                    cols.append(f'Sample')
                count+=1
                continue
            cols.append(column)
        tb4.columns = cols
        tb4.drop('Sample_R', inplace=True, axis=1, errors='ignore')
        
        cols = []
        count = 1
        for column in tb4.columns:
            if column == 'Area':
                if  count > 1:
                    cols.append(f'Area_c{count}')
                else:
                    cols.append(f'Area')
                count+=1
                continue
            cols.append(column)
        tb4.columns = cols
        cols = []
        count = 1
        for column in tb4.columns:
            if column == 'DAPIMean':
                if  count > 1:
                    cols.append(f'DAPIMean_c{count}')
                else:
                    cols.append(f'DAPIMean')
                count+=1
                continue
            cols.append(column)
        tb4.columns = cols
    
    else:
        tb4=tb1
        tb4.drop('Unnamed: 0', inplace=True, axis=1, errors='ignore')
        tb4.drop('Slice', inplace=True, axis=1, errors='ignore')
        tb4.drop('Num', inplace=True, axis=1, errors='ignore')
        tb4.drop('X', inplace=True, axis=1, errors='ignore')
        tb4.drop('Y', inplace=True, axis=1, errors='ignore')
    
    
    tb_tmp1='Before_Correction_'+Samples[numSamtb]+'.csv'
    filepath_tmp1 = Path(dir_save_data+'/Before_Correction/'+tb_tmp1)
    filepath_tmp1.parent.mkdir(parents=True, exist_ok=True)
    
    
    tb4.to_csv(filepath_tmp1)       
    print(Samples[numSamtb])        
       
print('Done!')
#winsound.Beep(frequency, duration)
    

B04
C04
D04
E04
F04
G04
Done!


In [6]:
########## PART 3 ###########
#### Making DATA Tables ######
###############################

for numSamtb in range(0, len(Samples)): ### range(0, len(Samples))
    
    data_file_name = 'Before_Correction_'+Samples[numSamtb]+'.csv'
    
    data_file_p = dir_save_data+'/Before_Correction/'+data_file_name
    
    
    data_file = pd.read_csv(data_file_p)
    
    colN = data_file.columns
    pos=3
    num_c=0
    for column in colN:
        if "Area" in column:
            num_c+=1
            
    Int_Dapi=data_file['Area']*data_file['DAPIMean']
    data_file.insert(loc=pos, column='Int_Dapi', value=Int_Dapi, allow_duplicates=True)
    
    for nc in range(0, num_c):
        num_w=0
        for column in colN:
            if "_c"+str(nc+1) in column:
                num_w+=1
        if nc>0:
            num_w=num_w-2
        
        for wc in range(0, num_w):
            if nc==0:
                Int=data_file['Area']*data_file['w'+str(wc+2)+'_c'+str(nc+1)]
            else:
                Int=data_file['Area_c'+str(nc+1)]*data_file['w'+str(wc+2)+'_c'+str(nc+1)]
            data_file.insert(loc=pos+1, column='Int_w'+str(wc+2)+'_c'+str(nc+1), value=Int, allow_duplicates=True)
            pos+=1
            
    data_file.drop('Unnamed: 0', inplace=True, axis=1, errors='ignore')  
    data_file.drop(data_file.iloc[:, pos:], inplace=True, axis=1)

    data_file_name_ps = 'Before_Correction_Int'+Samples[numSamtb]+'.csv'
    data_file_ps = Path(dir_save_data+'/Before_Correction_Int/'+data_file_name_ps)
    data_file_ps.parent.mkdir(parents=True, exist_ok=True)

    data_file.to_csv(data_file_ps)



In [8]:
########## PART 4 ###########
#### Background Calculation ######
###############################

# Background correction is calculated using designated control samples.

# Specify control samples
Ctrl_Samples = ['G04', 'F04', 'E04', 'D04']

# Assign control samples to cycles
Ctrl_for_cycle = ['c1:F04', 'c2:E04', 'c3:D04']

# Specify Wavelength/channel for correction
Ctrl_w = ['w2', 'w3', 'w4']


from scipy.stats import linregress

Samples_no_Ctrl = [i for i in Samples if i not in Ctrl_Samples]

h_for_tb_corr = [' ']*len(Ctrl_w)*2

sch=0
for numCtrlW in range (0, len(Ctrl_w)):
    h_for_tb_corr[sch]=Ctrl_w[numCtrlW]+'a'
    h_for_tb_corr[sch+1]=Ctrl_w[numCtrlW]+'b'
    sch+=2

tb_corr=pd.DataFrame(0, index=np.arange(len(Ctrl_for_cycle)), columns=h_for_tb_corr)

for numCtrlC in range (0, len(Ctrl_for_cycle)):
    Ctrl_file = Ctrl_for_cycle[numCtrlC].split(':')
    CycN =  int(Ctrl_file[0].replace('c', ''))
    data_tb1_p = dir_save_data+'/Before_Correction_Int/'+'Before_Correction_Int'+Ctrl_file[1]+'.csv'
    data_tb1 = pd.read_csv(data_tb1_p)
           
    for numCtrlW in range (0, len(Ctrl_w)):
        if CycN ==1:
            ValX=data_tb1['Area']
            ValY=data_tb1['Int_'+Ctrl_w[numCtrlW]+'_'+Ctrl_file[0]]
        else:
            ValX=data_tb1['Int_'+Ctrl_w[numCtrlW]+'_c'+str(CycN-1)]
            ValY=data_tb1['Int_'+Ctrl_w[numCtrlW]+'_'+Ctrl_file[0]]
        
        ##### Get rid of the 5% highest values before calculating the formula
        combXY = [ValX, ValY]
        combXY_tb1 = pd.concat(combXY, axis=1)
        combXY_tb2=combXY_tb1.sort_values('Int_'+Ctrl_w[numCtrlW]+'_'+Ctrl_file[0])
        NtoCut=round(len(combXY_tb2)*0.05)
        combXY_tb2.drop(combXY_tb2.tail(NtoCut).index, inplace = True)
        ##### END of Get rid of the 5% highest values
        
        slope, intercept, r_value, p_value, std_err  = linregress(combXY_tb2.iloc[:, 0], combXY_tb2.iloc[:, 1])
        
        
        col = Ctrl_w[numCtrlW] + 'a'
        tb_corr[col] = tb_corr[col].astype('Float64')
        tb_corr.at[numCtrlC, col] = round(slope, 2)
       
        col11 = Ctrl_w[numCtrlW] + 'b'
        tb_corr[col11] = tb_corr[col11].astype('Float64')
        tb_corr.at[numCtrlC, col11] = round(intercept, 2)
        
print (tb_corr)   

       w2a         w2b      w3a        w3b     w4a        w4b
0  3196.98   141243.01  1733.02   32186.23  1754.0   39785.16
1     0.55  1142026.41     0.27  126290.26    0.45  170661.71
2     0.89    98864.52     1.11  -11403.35    0.77   118739.3


In [40]:
########## PART 5 ###########
#### Final Results and Column Renaming ######
###############################

for numSamtb in range(0, len(Samples_no_Ctrl)): ### 
    
    data_file_name = 'Before_Correction_Int'+Samples_no_Ctrl[numSamtb]+'.csv'
    
    data_file_p = dir_save_data+'/Before_Correction_Int/'+data_file_name
    
    
    data_file = pd.read_csv(data_file_p)
    
    colN = data_file.columns
    pos=3 ### starting position for inserting new columns
    
    num_c=len(colN)-4
        
    for nc in range(0, num_c):
        num_w=0
        for column in colN:
            if "_c"+str(nc) in column:
                num_w+=1
        
        if num_w >0:
            for wc in range(0, num_w):
                
                if 'Int_w'+str(wc+2)+'_c'+str(nc) in colN:
                    not_corr_v = data_file['Int_w'+str(wc+2)+'_c'+str(nc)]
                    
                    if nc ==1:
                        bg_v = data_file['Area'] 
                
                    else:
                        bg_v = data_file['Int_w'+str(wc+2)+'_c'+str(nc-1)]
                        
                    if 'w'+str(wc+2)+'a' in h_for_tb_corr:
                        bg_val = bg_v*tb_corr['w'+str(wc+2)+'a'][nc-1]+tb_corr['w'+str(wc+2)+'b'][nc-1]
                    else:
                        bg_val=bg_v*0
                    
                    corr_v = not_corr_v - bg_val
                
                    data_file.insert(loc=pos+1, column='w'+str(wc+2)+'_c'+str(nc), value=corr_v, allow_duplicates=True)
                    pos+=1
            
    
    data_file.drop('Unnamed: 0', inplace=True, axis=1, errors='ignore')  
    data_file.drop(data_file.iloc[:, pos:], inplace=True, axis=1)

    data_file_name_ps = 'Final_'+Samples_no_Ctrl[numSamtb]+'.csv'
    data_file_ps = Path(dir_save_data+'/Final_DATA/'+data_file_name_ps)
    data_file_ps.parent.mkdir(parents=True, exist_ok=True)

    ######### Column names can be customized to reflect biological markers or proteins of interest.##########
      
    data_file.rename(columns = {'Int_Dapi':'Dapi', 'w2_c1':'YourProtein1','w3_c1':'YourProtein2', 'w4_c1':'YourProtein3',
                                'w2_c2':'YourProtein4', 'w3_c2':'YourProtein5', 'w4_c2':'YourProtein6',
                                'w2_c3':'YourProtein7', 'w3_c3':'YourProtein8', 'w4_c3':'YourProtein9' }, inplace = True)

     
    data_file.to_csv(data_file_ps)
    
    if numSamtb ==0:
       
        All_in_one_data_file = data_file
        
    else:
        All_in_one = [All_in_one_data_file, data_file]
        All_in_one_data_file = pd.concat(All_in_one, axis=0)
        
data_file_name_all = 'All_in_one_Final.csv'
data_file_all = Path(dir_save_data+'/Final_DATA/'+data_file_name_all)
data_file_all.parent.mkdir(parents=True, exist_ok=True)
        
All_in_one_data_file.to_csv(data_file_all)
print('Done!')

Done!
